# NumPy & Pandas: Side by Side
Exploring both libraries and transferring data between them.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

## 1. Creating Data

In [ ]:
# NumPy: arrays
arr = np.array([10, 20, 30, 40, 50])
print("1D array:", arr)

matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])
print("2D array:\n", matrix)

# Random data
random_data = np.random.randn(5, 3)
print("\nRandom 5x3:\n", random_data)

In [ ]:
# Pandas: DataFrames and Series
s = pd.Series([10, 20, 30, 40, 50], name="values")
print("Series:\n", s)

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "age": [25, 30, 35, 28, 32],
    "score": [88.5, 92.0, 76.3, 95.1, 81.7]
})
print("\nDataFrame:\n", df)

## 2. Basic Operations

In [ ]:
# NumPy: vectorized math (fast, element-wise)
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("a + b:", a + b)
print("a * b:", a * b)
print("a ** 2:", a ** 2)
print("sqrt(b):", np.sqrt(b))
print("dot product:", np.dot(a, b))

In [ ]:
# Pandas: column operations and filtering
df["score_curved"] = df["score"] * 1.05  # new column from math
df["pass"] = df["score"] >= 80            # boolean column

print("With new columns:\n", df)
print("\nFiltered (score > 85):\n", df[df["score"] > 85])

## 3. Aggregation & Stats

In [ ]:
# NumPy
data = np.random.randn(1000)

print("mean:", np.mean(data))
print("std: ", np.std(data))
print("min: ", np.min(data))
print("max: ", np.max(data))
print("25th/50th/75th percentiles:", np.percentile(data, [25, 50, 75]))

In [ ]:
# Pandas: .describe() gives you all of this at once
print(df[["age", "score"]].describe())

# Groupby (pandas strength)
df["group"] = ["A", "B", "A", "B", "A"]
print("\nMean score by group:\n", df.groupby("group")["score"].mean())

## 4. Indexing & Slicing

In [ ]:
# NumPy: integer and boolean indexing
m = np.arange(20).reshape(4, 5)
print("Matrix:\n", m)
print("Row 1:", m[1])
print("Col 2:", m[:, 2])
print("Submatrix (rows 0-1, cols 1-3):\n", m[0:2, 1:4])
print("Elements > 10:", m[m > 10])  # boolean mask

In [ ]:
# Pandas: loc (label), iloc (integer), and boolean indexing
print("By label (loc):")
print(df.loc[0:2, ["name", "score"]])

print("\nBy position (iloc):")
print(df.iloc[0:2, 0:3])

print("\nBoolean filter:")
print(df.loc[df["age"] > 29, ["name", "age"]])

## 5. Transferring Between NumPy and Pandas
This is where it gets fun — they interop seamlessly for numerical data.

In [ ]:
# --- NumPy → Pandas ---

# Start with a numpy array
measurements = np.random.randn(100, 3) * 10 + 50  # 100 samples, 3 sensors
print("NumPy shape:", measurements.shape)
print("dtype:", measurements.dtype)

# Convert to DataFrame (add labels!)
df_sensors = pd.DataFrame(measurements, columns=["sensor_a", "sensor_b", "sensor_c"])
print("\nAs DataFrame:")
df_sensors.head()

In [ ]:
# Now use pandas tools on the data
print("Quick stats:")
print(df_sensors.describe().round(2))

print("\nCorrelation matrix:")
print(df_sensors.corr().round(3))

In [ ]:
# --- Pandas → NumPy ---

# Pull out as numpy array for computation
arr_back = df_sensors.to_numpy()  # or df_sensors.values
print("Back to NumPy:", type(arr_back), arr_back.shape)

# Single column → 1D array
col_a = df_sensors["sensor_a"].to_numpy()
print("Single column:", col_a.shape)

# Now do numpy-style math
normalized = (arr_back - arr_back.mean(axis=0)) / arr_back.std(axis=0)
print("\nNormalized (mean ≈ 0, std ≈ 1):")
print("  means:", normalized.mean(axis=0).round(4))
print("  stds: ", normalized.std(axis=0).round(4))

In [ ]:
# --- Round trip: numpy → pandas → manipulate → numpy ---

# Simulate: raw signal data as numpy
t = np.linspace(0, 2 * np.pi, 200)
signal = np.sin(t) + 0.3 * np.random.randn(200)

# Put into pandas for analysis
df_signal = pd.DataFrame({"time": t, "raw": signal})
df_signal["smoothed"] = df_signal["raw"].rolling(window=10, center=True).mean()
df_signal["residual"] = df_signal["raw"] - df_signal["smoothed"]

print(df_signal.head(15))

# Pull smoothed signal back to numpy for further math
smoothed_np = df_signal["smoothed"].dropna().to_numpy()
print(f"\nSmoothed array: shape={smoothed_np.shape}, dtype={smoothed_np.dtype}")
print(f"RMS of smoothed signal: {np.sqrt(np.mean(smoothed_np**2)):.4f}")

## 6. NumPy Linear Algebra (where NumPy really shines)

In [ ]:
# Matrix operations
A = np.array([[2, 1], [1, 3]])
b = np.array([5, 7])

# Solve Ax = b
x = np.linalg.solve(A, b)
print("Solution to Ax = b:", x)
print("Verify A @ x:", A @ x)

# Eigenvalues
eigenvalues, eigenvectors = np.linalg.eig(A)
print("\nEigenvalues:", eigenvalues)

# Matrix inverse and determinant
print("Inverse:\n", np.linalg.inv(A))
print("Determinant:", np.linalg.det(A))

## 7. Performance Comparison
NumPy is faster for raw numerical computation.

In [ ]:
import time

n = 1_000_000
np_arr = np.random.randn(n)
pd_ser = pd.Series(np_arr)

# NumPy sum
start = time.perf_counter()
for _ in range(100):
    np.sum(np_arr)
np_time = time.perf_counter() - start

# Pandas sum
start = time.perf_counter()
for _ in range(100):
    pd_ser.sum()
pd_time = time.perf_counter() - start

print(f"NumPy sum (100x): {np_time:.4f}s")
print(f"Pandas sum (100x): {pd_time:.4f}s")
print(f"Ratio: pandas is ~{pd_time/np_time:.1f}x slower")

## Takeaways

- **NumPy** for math, linear algebra, and raw numerical speed
- **Pandas** for labeled data, mixed types, groupby, merging, and exploratory analysis
- Transfer freely: `pd.DataFrame(numpy_array)` and `df.to_numpy()`
- In practice, use both together — pandas for wrangling, numpy for computation